# Regression – Task 7: Model Comparison

**Aufgabe:** Vergleich des in Task 5 optimierten Machine-Learning-Modells (Ridge Regression mit polynomialen Features) mit dem Regressionsbaum aus Task 6.

**Ziel:** Bestimmen, welches Modell besser performt und warum, anhand von Kennzahlen und den jeweiligen Modellstaerken bzw. -schwaechen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

plt.rc('font', size=12)
plt.rc('axes', labelsize=12, titlesize=12)
plt.rc('legend', fontsize=11)

---
### Schritt 1: Daten laden

In [ ]:
df = pd.read_csv('../data/dataset_cleaned.csv')
X = df.drop(['Churn', 'CLV_Continuous'], axis=1)
y = df['CLV_Continuous']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print('Trainingsdaten:', X_train.shape)
print('Testdaten:     ', X_test.shape)

---
### Schritt 2: Optimierte Modelle aus Task 5 und Task 6 aufsetzen

Verwendet werden die in den vorherigen Tasks ermittelten Bestkonfigurationen:
- **Task 5 (Ridge):** PolynomialFeatures Grad 2 + StandardScaler + Ridge mit $\alpha=1.0$
- **Task 6 (Tree):** `max_depth=15`, `min_samples_split=20`, `min_samples_leaf=5`, `max_features=None`, `ccp_alpha=0.0`

In [ ]:
ridge_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])

tree_model = DecisionTreeRegressor(
    random_state=RANDOM_STATE,
    max_depth=15,
    min_samples_split=20,
    min_samples_leaf=5,
    max_features=None,
    ccp_alpha=0.0
)

ridge_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)

---
### Schritt 3: Performance vergleichen

In [ ]:
def get_metrics(model, X_train, y_train, X_test, y_test):
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    train_r2 = r2_score(y_train, pred_train)
    test_r2 = r2_score(y_test, pred_test)
    return {
        'Train MAE': mean_absolute_error(y_train, pred_train),
        'Test MAE': mean_absolute_error(y_test, pred_test),
        'Train RMSE': np.sqrt(mean_squared_error(y_train, pred_train)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, pred_test)),
        'Train R2': train_r2,
        'Test R2': test_r2,
        'R2-Gap': train_r2 - test_r2
    }

ridge_metrics = get_metrics(ridge_model, X_train, y_train, X_test, y_test)
tree_metrics = get_metrics(tree_model, X_train, y_train, X_test, y_test)

comparison = pd.DataFrame([
    ['Ridge Regression (Task 5)', *ridge_metrics.values()],
    ['Regression Tree (Task 6)', *tree_metrics.values()]
], columns=['Modell', 'Train MAE', 'Test MAE', 'Train RMSE', 'Test RMSE', 'Train R2', 'Test R2', 'R2-Gap'])

comparison = comparison.set_index('Modell')
comparison

In [ ]:
# Kompakter Plot fuer Testmetriken
plot_df = comparison[['Test MAE', 'Test RMSE', 'Test R2']].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_df['Test MAE'].plot(kind='bar', ax=axes[0], color=['steelblue', 'darkorange'])
axes[0].set_title('Test MAE (niedriger ist besser)')
axes[0].tick_params(axis='x', rotation=20)

plot_df['Test RMSE'].plot(kind='bar', ax=axes[1], color=['steelblue', 'darkorange'])
axes[1].set_title('Test RMSE (niedriger ist besser)')
axes[1].tick_params(axis='x', rotation=20)

plot_df['Test R2'].plot(kind='bar', ax=axes[2], color=['steelblue', 'darkorange'])
axes[2].set_title('Test R2 (hoeher ist besser)')
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

---
### Schritt 4: Diskussion – welches Modell ist besser und warum?

**Kurzentscheidung (nach Testleistung):** Das Modell mit dem niedrigeren Test-RMSE und hoeheren Test-$R^2$ performt insgesamt besser.

**Begruendung ueber Modellstaerken/-schwaechen:**
- **Ridge Regression (Task 5):**
  - Staerke: Sehr stabil, gute Generalisierung bei korrelierten/polynomialen Features durch Regularisierung.
  - Schwaeche: Kann stark nichtlineare, stueckweise Zusammenhaenge weniger flexibel abbilden als Baeume.
- **Regression Tree (Task 6):**
  - Staerke: Modelliert nichtlineare Beziehungen und Interaktionen direkt, oft sehr gute Vorhersageguete.
  - Schwaeche: Hoehere Overfitting-Gefahr, deshalb auf Tuning (Tiefe, Leaf-Größe, Split-Regeln) angewiesen.

**Fazit:**
- Wenn der Regressionsbaum die besseren Testmetriken erzielt, ist er hier die beste Wahl fuer maximale Vorhersageguete.
- Wenn Ridge sehr aehnliche Testwerte bei kleinerem Gap zeigt, kann Ridge die robustere Wahl fuer Stabilitaet und Interpretierbarkeit sein.

# Regression – Task 7: Model Comparison

**Ziel:** Vergleich des in Task 5 optimierten Machine-Learning-Modells (**Ridge Regression mit polynomialen Features, Grad 2**) mit dem in Task 6 optimierten **Regression Tree Model**. Anschliessend wird begruendet, welches Modell besser ist und warum.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

plt.rc('font', size=13)
plt.rc('axes', labelsize=13, titlesize=13)
plt.rc('legend', fontsize=12)

---
### Schritt 1: Daten laden und gemeinsamen Split erzeugen

Damit der Vergleich fair ist, werden beide Modelle auf demselben Trainings-/Test-Split trainiert und bewertet.

In [ ]:
df = pd.read_csv('../data/dataset_cleaned.csv')
X = df.drop(['Churn', 'CLV_Continuous'], axis=1)
y = df['CLV_Continuous']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print('Trainingsdaten:', X_train.shape)
print('Testdaten:     ', X_test.shape)

---
### Schritt 2: Optimierte Modelle aus Task 5 und Task 6 neu trainieren

- **Task 5:** Ridge Regression mit polynomialen Features (Grad 2) und GridSearchCV fuer $\alpha$
- **Task 6:** DecisionTreeRegressor mit GridSearchCV ueber die Baumkomplexitaet

In [ ]:
# Task 5: Optimierte Ridge Regression
ridge_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge', Ridge())
])

ridge_grid = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid={'ridge__alpha': np.logspace(-4, 4, 17)},
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
    refit=True
)
ridge_grid.fit(X_train, y_train)
best_ridge_model = ridge_grid.best_estimator_

# Task 6: Optimierter Regressionsbaum
tree_param_grid = {
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': [None, 'sqrt', 'log2'],
    'ccp_alpha': [0.0, 0.0001, 0.001, 0.01, 0.1]
}

tree_grid = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_grid=tree_param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
    refit=True
)
tree_grid.fit(X_train, y_train)
best_tree_model = tree_grid.best_estimator_

print('Bestes Ridge-alpha:', ridge_grid.best_params_['ridge__alpha'])
print('Bestes Tree-Setup :', tree_grid.best_params_)

---
### Schritt 3: Modelle mit denselben Metriken vergleichen

Es werden **MAE**, **RMSE**, **$R^2$**, **Train-Test-Gap** und der **CV-RMSE** verglichen.

- **MAE:** durchschnittlicher absoluter Fehler
- **RMSE:** gewichtet grosse Fehler staerker
- **$R^2$:** erklaerte Varianz
- **Train-Test-Gap:** Hinweis auf Overfitting
- **CV-RMSE:** Stabilitaet auf Validierungsdaten

In [ ]:
def evaluate_model(model, model_name, cv_rmse):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)

    return {
        'Modell': model_name,
        'Train MAE': mean_absolute_error(y_train, y_pred_train),
        'Test MAE': mean_absolute_error(y_test, y_pred_test),
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'Train R2': train_r2,
        'Test R2': test_r2,
        'R2-Gap': train_r2 - test_r2,
        'CV-RMSE': cv_rmse
    }

ridge_results = evaluate_model(best_ridge_model, 'Optimized Ridge Regression', -ridge_grid.best_score_)
tree_results = evaluate_model(best_tree_model, 'Optimized Regression Tree', -tree_grid.best_score_)

comparison_df = pd.DataFrame([ridge_results, tree_results]).set_index('Modell')
pd.set_option('display.float_format', '{:.4f}'.format)
comparison_df

---
### Schritt 4: Visualisierung des Modellvergleichs

Zur besseren Einordnung werden die wichtigsten Testmetriken direkt gegenuebergestellt.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

comparison_df['Test RMSE'].plot(kind='bar', ax=axes[0], color=['steelblue', 'darkorange'])
axes[0].set_title('Test RMSE')
axes[0].set_ylabel('RMSE')
axes[0].tick_params(axis='x', rotation=20)

comparison_df['Test MAE'].plot(kind='bar', ax=axes[1], color=['steelblue', 'darkorange'])
axes[1].set_title('Test MAE')
axes[1].set_ylabel('MAE')
axes[1].tick_params(axis='x', rotation=20)

comparison_df['Test R2'].plot(kind='bar', ax=axes[2], color=['steelblue', 'darkorange'])
axes[2].set_title('Test $R^2$')
axes[2].set_ylabel('$R^2$')
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

---
### Schritt 5: Interpretation und Schlussfolgerung

#### Welche Modelle sind in welchen Aspekten besser?
- Der **optimierte Regressionsbaum** erzielt die bessere **Testleistung** bei **RMSE** und **$R^2$**.
- Die **optimierte Ridge Regression** erzielt den besseren **MAE**, den kleineren **Train-Test-Gap** und den deutlich besseren **CV-RMSE**.

#### Warum schneidet der Regressionsbaum bei RMSE und $R^2$ besser ab?
- Entscheidungsbaeume koennen **nichtlineare Zusammenhaenge** und **komplexe Interaktionen** direkt modellieren.
- In diesem Datensatz scheint der CLV stark durch nichtlineare Schwellen und Segmentierungen beeinflusst zu sein, was dem Baum entgegenkommt.
- Die hohe Bedeutung von `CBalance` und `CEstimatedSalary` zeigt, dass wenige dominante Split-Variablen bereits viel Struktur erklaeren.

#### Warum ist Ridge trotzdem stabiler?
- Ridge Regression ist ein **glatteres und robusteres Modell**.
- Der kleinere Train-Test-Gap und der niedrigere CV-RMSE zeigen, dass Ridge auf verschiedenen Folds **stabiler generalisiert**.
- Das spricht dafuer, dass der Regressionsbaum trotz besserer Testwerte noch etwas staerker zur Varianz neigt.

#### Schlussfolgerung
- **Nach reiner Testleistung** ist der **optimierte Regressionsbaum** das bessere Modell.
- **Nach Robustheit und Stabilitaet** ist die **optimierte Ridge Regression** die sicherere Wahl.
- Wenn der Fokus auf der bestmoeglichen Vorhersage auf diesem Testsplit liegt, ist der Regressionsbaum vorzuziehen. Wenn dagegen ein stabileres und besser kontrollierbares Modell gewuenscht ist, spricht mehr fuer Ridge Regression.